# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Loading the dataset**

In [1]:
import duckdb
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN '')")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [4]:
fact_table = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #1: "Random Forest beats the hand-written rule under client-holdout, reversing what
in-sample scoring showed."** (Notebook 02, starter CSV: hand rule 0.300/0.400 vs. tree
0.400/0.440 at Precision@20/@50 on held-out clients — the opposite ranking from the in-sample
comparison. Notebook 06 repeated this shape on the real warehouse.)

- **Where does the label come from?** On the starter CSV: `is_declining_label`, derived from
  `trend_direction`, itself derived from `trend_pct` — a same-window proxy, flagged as
  circular back in Week 1. On the warehouse (Notebook 06): `clicks_mar < 0.8 × clicks_feb`, a
  genuine two-window comparison, which is a real improvement — but Notebook 05 also found that
  this exact definition mechanically floors decline rate at 0 for any zero-click-Feb page. So
  even the "fixed" label still needs the `clicks_feb > 0` restriction to be trustworthy, which
  Notebook 06 applied — worth stating explicitly rather than assuming the label is clean just
  because it's two-window.
- **Does the validation design carry the claim?** Client-holdout is the right call and is a
  real strength — it directly answers "would this generalize to a client the model has never
  seen," which in-sample scoring cannot. But the claim should stay modest: on the starter CSV,
  both scores were still low in absolute terms (0.30–0.44), so "the tree wins" is a real but
  narrow result — a ranking edge on a small, noisy metric, not a claim that either approach is
  good enough to deploy unsupervised.

**Finding #2: "CTR increases monotonically with position tier on real warehouse data"**
(Notebook 05, Section 3: mean CTR `deep` 0.0027 → `page_2` 0.0031 → `page_1` 0.0053 →
`top_3` 0.0109.)

- **Where does the label come from?** No proxy label here at all — CTR is a directly measured
  ratio (`clicks_feb / impressions_feb`), so this finding doesn't inherit the label-circularity
  risk that Finding #1 does. That's a genuine strength worth naming, not just a caveat.
- **Does the validation design carry the claim?** No — and this is the real methodology gap.
  This was a single mid-panel month (February), read as one static table, with no train/test
  split and no check against a different month or a different client mix. Notebook 05 also
  found that `decline_rate` climbs with position tier in the exact same table, and flagged that
  as a likely label-floor artifact (well-positioned pages simply have more clicks to lose). The
  CTR-by-tier pattern itself is probably real and robust — it matches the starter-CSV finding
  from Week 1 — but "probably real, and previously replicated once" is different from
  "validated," and the writeup should say which one it actually is.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
FEATURE_MONTH = "2026-02"

# --- Finding #1 check: historical numbers, cited not re-derived ---
# Per instruction, the starter CSV (data/raw/content_refresh_anonymized.csv) is not reopened
# in this notebook -- these are the exact numbers already verified and reported in Notebook 02.
print("Finding #1 -- Notebook 02, starter CSV, client-holdout (historical, not re-run here):")
print(f"{'Model':<20}{'Precision@20':>15}{'Precision@50':>15}")
print(f"{'Hand rule':<20}{0.300:>15.3f}{0.400:>15.3f}")
print(f"{'Decision tree':<20}{0.400:>15.3f}{0.440:>15.3f}")
print("Tree wins at both K on held-out clients -- the reverse of the in-sample comparison.")
print("(Warehouse repetition of this comparison lives in Notebook 06, Section 3 of this")
print(" notebook re-runs it under BOTH a naive and a client-grouped split for direct before/after.)")

# --- Finding #2 check: re-run fresh against real warehouse data, not cited from memory ---
print("\nFinding #2 -- CTR by position tier, re-queried live on real warehouse data:")
finding2 = con.sql(f"""
    WITH feb AS (
        SELECT content_hash_id,
               AVG(gsc_avg_position) AS avg_position_feb,
               SUM(gsc_clicks)*1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr_feb
        FROM read_parquet('{fact_table}')
        WHERE month = '{FEATURE_MONTH}'
        GROUP BY content_hash_id
    )
    SELECT CASE
             WHEN avg_position_feb IS NULL THEN 'no_position'
             WHEN avg_position_feb <= 3  THEN 'top_3'
             WHEN avg_position_feb <= 10 THEN 'page_1'
             WHEN avg_position_feb <= 20 THEN 'page_2'
             ELSE 'deep' END AS position_tier,
           COUNT(*) AS n,
           ROUND(AVG(ctr_feb), 4) AS mean_ctr
    FROM feb
    WHERE avg_position_feb IS NOT NULL
    GROUP BY 1
    ORDER BY mean_ctr DESC
""")
print(finding2)
print("\nCheck: does mean_ctr still climb monotonically top_3 > page_1 > page_2 > deep,")
print("matching Notebook 05's reported 0.0109 / 0.0053 / 0.0031 / 0.0027?")

Finding #1 -- Notebook 02, starter CSV, client-holdout (historical, not re-run here):
Model                  Precision@20   Precision@50
Hand rule                     0.300          0.400
Decision tree                 0.400          0.440
Tree wins at both K on held-out clients -- the reverse of the in-sample comparison.
(Warehouse repetition of this comparison lives in Notebook 06, Section 3 of this
 notebook re-runs it under BOTH a naive and a client-grouped split for direct before/after.)

Finding #2 -- CTR by position tier, re-queried live on real warehouse data:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────┬───────┬──────────┐
│ position_tier │   n   │ mean_ctr │
│    varchar    │ int64 │  double  │
├───────────────┼───────┼──────────┤
│ top_3         │ 19243 │   0.0106 │
│ page_1        │ 75898 │   0.0052 │
│ page_2        │ 31699 │   0.0031 │
│ deep          │ 26719 │   0.0025 │
└───────────────┴───────┴──────────┘


Check: does mean_ctr still climb monotonically top_3 > page_1 > page_2 > deep,
matching Notebook 05's reported 0.0109 / 0.0053 / 0.0031 / 0.0027?


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Split chosen: client-grouped**, same reasoning as every prior notebook — a client's own
pages share client-level idiosyncrasies (industry, content strategy, baseline traffic), so a
naive random row split lets a client's own pages leak between train and test even though no
single row is duplicated. That's a subtler leak than the Week-3 label-derived-column trap, but
the same family of problem: the split itself, not just the features, can be dishonest.

**Before/after, same panel, same features, same models, same metric** — only the split
changes.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
FEATURE_MONTH, LABEL_MONTH = "2026-02", "2026-03"

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Rebuild the Week-6 panel exactly (Feb features, Mar label, corrected decline definition)
panel = con.sql(f"""
    WITH feb AS (
        SELECT client_hash_id, content_hash_id,
               AVG(gsc_avg_position)      AS avg_position_feb,
               SUM(gsc_impressions)       AS impressions_feb,
               SUM(gsc_clicks)            AS clicks_feb,
               SUM(ga4_sessions)          AS ga4_sessions_feb,
               SUM(ga4_engaged_sessions)  AS ga4_engaged_sessions_feb,
               SUM(ga4_pageviews)         AS ga4_pageviews_feb,
               SUM(scroll_events)         AS scroll_events_feb,
               SUM(sessions_ai)           AS sessions_ai_feb
        FROM read_parquet('{fact_table}') WHERE month = '{FEATURE_MONTH}'
        GROUP BY client_hash_id, content_hash_id
    ), mar AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_mar
        FROM read_parquet('{fact_table}') WHERE month = '{LABEL_MONTH}'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT feb.*, mar.clicks_mar
    FROM feb JOIN mar USING (client_hash_id, content_hash_id)
    WHERE feb.clicks_feb > 0
""").df()

panel["is_declining"] = (panel["clicks_mar"] < 0.8 * panel["clicks_feb"]).astype(int)
panel["ctr_feb"] = panel["clicks_feb"] / panel["impressions_feb"].replace(0, np.nan)
panel["had_ga4_feb"] = panel["ga4_sessions_feb"] > 0
panel["engagement_rate_feb"] = np.where(panel["had_ga4_feb"], panel["ga4_engaged_sessions_feb"] / panel["ga4_sessions_feb"], 0.0)
panel["scroll_rate_feb"] = np.where(panel["ga4_pageviews_feb"] > 0, panel["scroll_events_feb"] / panel["ga4_pageviews_feb"], 0.0)
panel["ai_referral_share_feb"] = np.where(panel["had_ga4_feb"], panel["sessions_ai_feb"] / panel["ga4_sessions_feb"], 0.0)
numeric_cols = panel.select_dtypes(include=[np.number]).columns
panel[numeric_cols] = panel[numeric_cols].fillna(0)

feature_cols = ["avg_position_feb", "impressions_feb", "ctr_feb", "engagement_rate_feb", "scroll_rate_feb", "ai_referral_share_feb"]

def score_split(train_mask, test_mask, label):
    X_train, X_test = panel.loc[train_mask, feature_cols], panel.loc[test_mask, feature_cols]
    y_train, y_test = panel.loc[train_mask, "is_declining"].values, panel.loc[test_mask, "is_declining"].values
    lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE).fit(X_train, y_train)
    rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=RANDOM_STATE).fit(X_train, y_train)
    lr_scores = lr.predict_proba(X_test)[:, 1]
    rf_scores = rf.predict_proba(X_test)[:, 1]
    print(f"\n--- {label} (n_test={len(y_test)}) ---")
    print(f"{'Model':<20}{'Precision@20':>15}{'Precision@50':>15}")
    for name, scores in [("Logistic Regression", lr_scores), ("Random Forest", rf_scores)]:
        print(f"{name:<20}{precision_at_k(scores, y_test, 20):>15.3f}{precision_at_k(scores, y_test, 50):>15.3f}")

# BEFORE: naive random row split -- ignores client, lets a client's own pages leak both ways
rng = np.random.default_rng(RANDOM_STATE)
shuffled_idx = rng.permutation(panel.index)
n_test_rows = int(len(panel) * 0.2)
naive_test_mask = panel.index.isin(shuffled_idx[:n_test_rows])
naive_train_mask = ~naive_test_mask
score_split(naive_train_mask, naive_test_mask, "BEFORE -- naive random split (client leakage possible)")

# AFTER: client-grouped split -- same panel, same features, same models, only the split changes
unique_clients = panel["client_hash_id"].unique()
shuffled_clients = rng.permutation(unique_clients)
n_test_clients = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:n_test_clients])
grouped_test_mask = panel["client_hash_id"].isin(test_clients)
grouped_train_mask = ~grouped_test_mask
score_split(grouped_train_mask, grouped_test_mask, "AFTER -- client-grouped split (honest)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- BEFORE -- naive random split (client leakage possible) (n_test=10747) ---
Model                  Precision@20   Precision@50
Logistic Regression           0.950          0.960
Random Forest                 1.000          0.980

--- AFTER -- client-grouped split (honest) (n_test=7039) ---
Model                  Precision@20   Precision@50
Logistic Regression           0.850          0.840
Random Forest                 0.900          0.900


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Same attack as Week 3 and Week 4, run again on the final Week-6 feature set
(`avg_position_feb`, `impressions_feb`, `ctr_feb`, `engagement_rate_feb`, `scroll_rate_feb`,
`ai_referral_share_feb`). Every one of these six is built from `WHERE month = FEATURE_MONTH`
only — none touch `LABEL_MONTH`. This cell doesn't just assert that; it proves it by injecting
a genuinely label-derived column and confirming the score jumps, the same falsifiable test used
every prior week.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier


X_honest = panel.loc[grouped_train_mask, feature_cols]
y_train = panel.loc[grouped_train_mask, "is_declining"].values
X_honest_test = panel.loc[grouped_test_mask, feature_cols]
y_test = panel.loc[grouped_test_mask, "is_declining"].values
X_train_grouped = panel.loc[grouped_train_mask, feature_cols]
y_train_grouped = panel.loc[grouped_train_mask, "is_declining"].values

lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE).fit(X_train_grouped, y_train_grouped)
rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=RANDOM_STATE).fit(X_train_grouped, y_train_grouped)

t_honest = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=RANDOM_STATE).fit(X_honest, y_train)
p50_honest = precision_at_k(t_honest.predict_proba(X_honest_test)[:, 1], y_test, 50)
print(f"HONEST (6 features)      Precision@50: {p50_honest:.3f}")

# THE ATTACK: add clicks_mar, which is literally used to build the label
leaky_cols = feature_cols + ["clicks_mar"]
X_leaky = panel.loc[grouped_train_mask, leaky_cols]
X_leaky_test = panel.loc[grouped_test_mask, leaky_cols]
t_leaky = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=RANDOM_STATE).fit(X_leaky, y_train)
p50_leaky = precision_at_k(t_leaky.predict_proba(X_leaky_test)[:, 1], y_test, 50)
print(f"LEAKY (+ clicks_mar)     Precision@50: {p50_leaky:.3f}  <- should jump toward 1.0")

print(f"\nJump: {p50_leaky - p50_honest:+.3f}. The honest {p50_honest:.3f} is what stays in the model card.")
del X_leaky, X_leaky_test, t_leaky


HONEST (6 features)      Precision@50: 0.840
LEAKY (+ clicks_mar)     Precision@50: 1.000  <- should jump toward 1.0

Jump: +0.160. The honest 0.840 is what stays in the model card.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Boldest sentence I could imagine writing after seeing these numbers:**

> "My Random Forest model achieves 90% precision at the top 50 pages, proving it reliably
> identifies declining content and is ready for production use across all clients."

**Rewritten in safe language:**

> "Under a client-grouped split — the honest version, where no client's own pages appear in
> both train and test — Random Forest reached Precision@50 of 0.900 and Precision@20 of 0.900,
> both above Logistic Regression (0.840 / 0.850) on the same panel. That's an observed,
> directional result on this specific Feb→March window, not a guarantee that generalizes to
> every client or every month. The same model scored even higher under a naive random split
> (0.980 Precision@50) — a measured 8-point drop once client leakage was removed, which is a
> reminder that a single strong number, on its own, doesn't say whether it's trustworthy. This
> is decision-support for a reviewer's queue: a page landing in the top 50 is a candidate worth
> a human look, not an automatic action."

**What changed, word by word:**
- "proving it reliably identifies" → "an observed, directional result" — removes the causal/
  certainty claim; a precision score on one held-out split is evidence, not proof.
- "achieves 90% precision" (bare) → "0.900 ... under a client-grouped split" — the number
  alone is meaningless without saying *which* split produced it, especially now that the same
  model scored 0.980 under a weaker one. Stating the split is part of stating the claim
  honestly, not an optional detail.
- "ready for production use across all clients" → "a candidate worth a human look, not an
  automatic action" — matches the actual unit-of-analysis and decision framing from Week 1
  (a reviewer opens the page; the model doesn't act alone), and drops the unsupported
  generalization to "all clients," which was never tested — only ~20% of clients were held out
  once, not validated repeatedly across different client splits.
- Added the naive-vs-grouped gap (0.980 → 0.900) explicitly, rather than only reporting the
  favorable number — the leakage audit in Section 3 (0.840 → 1.000 honest-vs-leaky) makes the
  same point even more

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify every number cited in the Section 4 claim rewrite actually matches
# what Sections 2 and 3 computed -- a claim's numbers should be traceable to
# real output, not retyped from memory into a sentence.

claimed_numbers = {
    "rf_grouped_p50":    0.900,
    "rf_grouped_p20":    0.900,
    "lr_grouped_p50":    0.840,
    "lr_grouped_p20":    0.850,
    "rf_naive_p50":      0.980,
    "honest_p50":        0.840,
    "leaky_p50":         1.000,
    "leakage_jump":      0.160,
}

# Recompute the same four scores fresh, from the same panel/split objects already
# in memory from Sections 2 and 3, rather than trusting the printed text output.
rf_grouped_scores = rf.predict_proba(panel.loc[grouped_test_mask, feature_cols])[:, 1]
lr_grouped_scores = lr.predict_proba(panel.loc[grouped_test_mask, feature_cols])[:, 1]
y_test_grouped = panel.loc[grouped_test_mask, "is_declining"].values

recomputed = {
    "rf_grouped_p50": precision_at_k(rf_grouped_scores, y_test_grouped, 50),
    "rf_grouped_p20": precision_at_k(rf_grouped_scores, y_test_grouped, 20),
    "lr_grouped_p50": precision_at_k(lr_grouped_scores, y_test_grouped, 50),
    "lr_grouped_p20": precision_at_k(lr_grouped_scores, y_test_grouped, 20),
    "honest_p50": p50_honest,
    "leaky_p50": p50_leaky,
    "leakage_jump": p50_leaky - p50_honest,
}

print(f"{'Metric':<20}{'Claimed':>10}{'Recomputed':>12}{'Match?':>10}")
for k, claimed in claimed_numbers.items():
    if k in recomputed:
        actual = recomputed[k]
        match = abs(actual - claimed) < 0.005
        print(f"{k:<20}{claimed:>10.3f}{actual:>12.3f}{'OK' if match else 'MISMATCH':>10}")

print("\nNote: rf_naive_p50 (0.980) isn't re-checkable here -- the naive-split model")
print("wasn't kept in memory (only the grouped-split rf/lr were refit above). If that")
print("number matters to the claim, re-run Section 2's naive-split branch and store it")
print("in a named variable instead of only printing it, so it can be checked here too.")


Metric                 Claimed  Recomputed    Match?
rf_grouped_p50           0.900       0.900        OK
rf_grouped_p20           0.900       0.900        OK
lr_grouped_p50           0.840       0.840        OK
lr_grouped_p20           0.850       0.850        OK
honest_p50               0.840       0.840        OK
leaky_p50                1.000       1.000        OK
leakage_jump             0.160       0.160        OK

Note: rf_naive_p50 (0.980) isn't re-checkable here -- the naive-split model
wasn't kept in memory (only the grouped-split rf/lr were refit above). If that
number matters to the claim, re-run Section 2's naive-split branch and store it
in a named variable instead of only printing it, so it can be checked here too.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.